In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hoomd

def measure_pair_potential(
    pair_force,
    r_values,
    box_length=30,
    particle_type="A",
    seed=1,
):
    """
    Measure pair potential energy U(r) for two particles using a HOOMD pair force.
    """

    energies = []

    for r in r_values:
        # ----------------------------
        # Two-particle snapshot
        # ----------------------------
        snap = hoomd.Snapshot()
        snap.particles.N = 2
        snap.particles.types = [particle_type]
        snap.particles.typeid[:] = [0, 0]

        snap.configuration.box = [box_length, box_length, box_length, 0, 0, 0]

        snap.particles.position[:] = [
            [-r / 2, 0, 0],
            [ r / 2, 0, 0],
        ]

        # ----------------------------
        # Small temporary simulation
        # ----------------------------
        sim_test = hoomd.Simulation(
            device=hoomd.device.CPU(),
            seed=seed,
        )

        sim_test.create_state_from_snapshot(snap)

        integrator = hoomd.md.Integrator(dt=0.005)
        integrator.forces.append(pair_force)
        sim_test.operations.integrator = integrator

        thermo = hoomd.md.compute.ThermodynamicQuantities(
            filter=hoomd.filter.All()
        )
        sim_test.operations.computes.append(thermo)

        sim_test.run(0)

        energies.append(thermo.potential_energy)

    return np.array(energies)

In [ ]:
# ============================================================
# Parameters
# ============================================================

r_cut = 2.5
r_on = 2

epsilon = 1.0
sigma = 1.0

r_values = np.linspace(0.85, 3.0, 700)

# ============================================================
# mode = none
# ============================================================

cell_none = hoomd.md.nlist.Cell(buffer=0.4)

lj_none = hoomd.md.pair.LJ(
    nlist=cell_none,
    default_r_cut=r_cut,
    mode="none",
)

lj_none.params[("A", "A")] = dict(
    epsilon=epsilon,
    sigma=sigma,
)

# ============================================================
# mode = shift
# ============================================================

cell_shift = hoomd.md.nlist.Cell(buffer=0.4)

lj_shift = hoomd.md.pair.LJ(
    nlist=cell_shift,
    default_r_cut=r_cut,
    mode="shift",
)

lj_shift.params[("A", "A")] = dict(
    epsilon=epsilon,
    sigma=sigma,
)

# ============================================================
# mode = xplor
# ============================================================

cell_xplor = hoomd.md.nlist.Cell(buffer=0.4)

lj_xplor = hoomd.md.pair.LJ(
    nlist=cell_xplor,
    default_r_cut=r_cut,
    mode="xplor",
)

lj_xplor.params[("A", "A")] = dict(
    epsilon=epsilon,
    sigma=sigma,
)

lj_xplor.r_on[("A", "A")] = r_on

# ============================================================
# ForceShiftedLJ
# ============================================================

cell_fs = hoomd.md.nlist.Cell(buffer=0.4)

fs_lj = hoomd.md.pair.ForceShiftedLJ(
    nlist=cell_fs,
    default_r_cut=r_cut,
)

fs_lj.params[("A", "A")] = dict(
    epsilon=epsilon,
    sigma=sigma,
)

# ============================================================
# Measure all potentials
# ============================================================

U_none = measure_pair_potential(
    pair_force=lj_none,
    r_values=r_values,
    box_length=30,
)

U_shift = measure_pair_potential(
    pair_force=lj_shift,
    r_values=r_values,
    box_length=30,
)

U_xplor = measure_pair_potential(
    pair_force=lj_xplor,
    r_values=r_values,
    box_length=30,
)

U_force_shifted = measure_pair_potential(
    pair_force=fs_lj,
    r_values=r_values,
    box_length=30,
)

# ============================================================
# Plot
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(r_values, U_none,          label="LJ (none)")
plt.plot(r_values, U_shift,         label="LJ (shift)")
plt.plot(r_values, U_xplor,         label="LJ (xplor)")
plt.plot(r_values, U_force_shifted, label="ForceShiftedLJ")

plt.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.axvline(
    r_cut,
    color="red",
    linestyle="--",
    linewidth=1,
    label=rf"$r_{{cut}}={r_cut}$",
)

plt.axvline(
    r_on,
    color="gray",
    linestyle=":",
    linewidth=1,
    label=rf"$r_{{on}}={r_on}$",
)

plt.xlabel(r"$r/\sigma$")
plt.ylabel(r"$U(r)$")

plt.title(
    "Comparison of HOOMD Lennard-Jones cutoff treatments"
)

plt.ylim(-1.5, 1.0)
plt.xlim(r_values.min(), r_values.max())

plt.legend()
plt.grid(alpha=0.3)

plt.show()